# EPS Production: M9 Glucose vs LB Medium
Compares EPS production capacity (UDPGPT proxy) and growth between M9 minimal medium and LB rich medium using iML1515.

In [24]:
from cobra.io import read_sbml_model
from cobra import Reaction
import pandas as pd

model = read_sbml_model('iML1515.xml')
print(model)

iML1515


## Add EPS Pathways
Three EPS pathways added from `model_investigationMA.ipynb`. These are not in the base iML1515 and must be added manually each session.

- **Cellulose** (bcsA): `udpg_c + cdigmp_c → cellulose_c`
- **Colanic acid** (Wca): `udcpgl_c + udpgal_c + udpglcur_c + 2×gdpfuc_c + pep_c → colacid_c` — this is the main EPS proxy and what drives UDPGPT flux
- **PNAG** (pgaB): exports periplasmic `puacgam_p` to extracellular

In [25]:
from cobra import Reaction, Metabolite

# ── Pathway 1: Cellulose (bcsA) ──────────────────────────────────────────────
cel_c = Metabolite('cellulose_c', compartment='c', formula='C6H10O5')
cel_e = Metabolite('cellulose_e', compartment='e', formula='C6H10O5')

r_celsynth = Reaction('CELSYNTH', lower_bound=0.0, upper_bound=1000.0)
r_celsynth.add_metabolites({
    model.metabolites.get_by_id('udpg_c'):   -1.0,
    model.metabolites.get_by_id('cdigmp_c'): -0.001,
    model.metabolites.get_by_id('udp_c'):     1.0,
    cel_c:                                    1.0,
})
r_celt  = Reaction('CELLULOSEt',    lower_bound=0.0, upper_bound=1000.0)
r_celt.add_metabolites({cel_c: -1.0, cel_e: 1.0})
r_celex = Reaction('EX_cellulose_e', lower_bound=0.0, upper_bound=1000.0)
r_celex.add_metabolites({cel_e: -1.0})

# ── Pathway 2: Colanic acid (Wca) ─────────────────────────────────────────────
# COLASYNTH consumes udcpgl_c (the UDPGPT product) — this is what drives UDPGPT flux
col_c = Metabolite('colacid_c', compartment='c', formula='C33H52O27', name='Colanic acid repeating unit')
col_e = Metabolite('colacid_e', compartment='e', formula='C33H52O27', name='Colanic acid (extracellular)')

r_cola = Reaction('COLASYNTH', lower_bound=0.0, upper_bound=1000.0, name='Colanic acid synthase (Wca)')
r_cola.add_metabolites({
    model.metabolites.get_by_id('udcpgl_c'):   -1.0,
    model.metabolites.get_by_id('udpgal_c'):   -1.0,
    model.metabolites.get_by_id('udpglcur_c'): -1.0,
    model.metabolites.get_by_id('gdpfuc_c'):   -2.0,
    model.metabolites.get_by_id('pep_c'):      -1.0,
    col_c:                                      1.0,
    model.metabolites.get_by_id('udcpp_c'):     1.0,
    model.metabolites.get_by_id('udp_c'):       2.0,
    model.metabolites.get_by_id('gdp_c'):       2.0,
    model.metabolites.get_by_id('pi_c'):        1.0,
})
r_colat  = Reaction('COLAIDt',       lower_bound=0.0, upper_bound=1000.0)
r_colat.add_metabolites({col_c: -1.0, col_e: 1.0})
r_colaex = Reaction('EX_colacid_e',  lower_bound=0.0, upper_bound=1000.0)
r_colaex.add_metabolites({col_e: -1.0})

# ── Pathway 3: PNAG export (pgaB) ────────────────────────────────────────────
# Most of the PNAG pathway already exists in iML1515 — only extracellular export is missing
pnag_e   = Metabolite('puacgam_e', compartment='e', name='PNAG (extracellular)')
r_pnagex1 = Reaction('PUACGAMex',    lower_bound=0.0, upper_bound=1000.0)
r_pnagex1.add_metabolites({model.metabolites.get_by_id('puacgam_p'): -1.0, pnag_e: 1.0})
r_pnagex2 = Reaction('EX_puacgam_e', lower_bound=0.0, upper_bound=1000.0)
r_pnagex2.add_metabolites({pnag_e: -1.0})

model.add_reactions([r_celsynth, r_celt, r_celex,
                     r_cola, r_colat, r_colaex,
                     r_pnagex1, r_pnagex2])

print(f"Model now has {len(model.reactions)} reactions")
print(f"CELSYNTH:  {model.reactions.get_by_id('CELSYNTH').reaction}")
print(f"COLASYNTH: {model.reactions.get_by_id('COLASYNTH').reaction}")
print(f"PUACGAMex: {model.reactions.get_by_id('PUACGAMex').reaction}")

Model now has 2720 reactions
CELSYNTH:  0.001 cdigmp_c + udpg_c --> cellulose_c + udp_c
COLASYNTH: 2.0 gdpfuc_c + pep_c + udcpgl_c + udpgal_c + udpglcur_c --> colacid_c + 2.0 gdp_c + pi_c + udcpp_c + 2.0 udp_c
PUACGAMex: puacgam_p --> puacgam_e


## Define Media

In [26]:
# M9 minimal medium with glucose
m9_medium = {
    'EX_glc__D_e': 10.0,    # glucose as sole carbon source
    'EX_o2_e':     20.0,
    'EX_nh4_e':    10.0,
    'EX_pi_e':     10.0,
    'EX_so4_e':    10.0,
    'EX_h2o_e':    1000.0,
    'EX_h_e':      1000.0,
    'EX_na1_e':    1000.0,
    'EX_k_e':      1000.0,
    'EX_mg2_e':    1000.0,
    'EX_ca2_e':    1000.0,
    'EX_cl_e':     1000.0,
    'EX_fe2_e':    1000.0,
    'EX_mn2_e':    1000.0,
    'EX_zn2_e':    1000.0,
    'EX_cu2_e':    1000.0,
    'EX_cobalt2_e':1000.0,
    'EX_mobd_e':   1000.0,
    'EX_ni2_e':    1000.0,
}

# LB rich medium — casein hydrolysate amino acids + yeast extract vitamins
lb_medium = {
    # salts / ions
    'EX_pi_e':      1000.0,
    'EX_ni2_e':     10000.0,
    'EX_so4_e':     10000.0,
    'EX_o2_e':      23.0,
    'EX_na1_e':     1000.0,
    'EX_cl_e':      1000.0,
    'EX_k_e':       1000.0,
    'EX_mg2_e':     1000.0,
    'EX_ca2_e':     1000.0,
    'EX_fe2_e':     1000.0,
    'EX_mn2_e':     1000.0,
    'EX_zn2_e':     1000.0,
    'EX_cu2_e':     1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_mobd_e':    1000.0,
    # amino acids (casein hydrolysate, proportional to composition)
    'EX_ala__L_e':  39.89,
    'EX_arg__L_e':  58.51,
    'EX_asn__L_e':  26.60,
    'EX_asp__L_e':  45.21,
    'EX_cys__L_e':  13.30,
    'EX_gln__L_e':  50.53,
    'EX_glu__L_e':  50.53,
    'EX_gly_e':     63.83,
    'EX_his__L_e':  26.60,
    'EX_ile__L_e':  42.55,
    'EX_leu__L_e':  95.74,
    'EX_lys__L_e':  63.83,
    'EX_met__L_e':  31.91,
    'EX_phe__L_e':  47.87,
    'EX_pro__L_e':  34.57,
    'EX_ser__L_e':  71.81,
    'EX_thr__L_e':  37.23,
    'EX_trp__L_e':  10.64,
    'EX_tyr__L_e':  39.89,
    'EX_val__L_e':  42.55,
    # vitamins (yeast extract)
    'EX_thm_e':     1000.0,
    'EX_nac_e':     1000.0,
    'EX_pnto__R_e': 1000.0,
    'EX_pydx_e':    1000.0,
    'EX_btn_e':     1000.0,
}

print('M9 components:', len(m9_medium))
print('LB components:', len(lb_medium))

M9 components: 19
LB components: 40


## EPS Analysis Function
Two-stage optimization:
1. Maximize biomass to find `mu_max`
2. Constrain biomass ≥ 90% of `mu_max`, then maximize `EX_colacid_e` (colanic acid secretion — the main EPS output)

All three EPS secretion fluxes are reported alongside the upstream pathway reactions.

In [27]:
BIOMASS_RXN = 'BIOMASS_Ec_iML1515_core_75p37M'
EPS_RXN     = 'EX_colacid_e'   # colanic acid secretion — main EPS output

# upstream pathway reactions + all three EPS secretion outputs
EPS_RXNS = [
    'UDPGPT',        # wcaJ — colanic acid initiation
    'COLASYNTH',     # Wca — colanic acid assembly
    'EX_colacid_e',  # colanic acid secretion
    'CELSYNTH',      # bcsA — cellulose synthesis
    'EX_cellulose_e',# cellulose secretion
    'EX_puacgam_e',  # PNAG secretion
    'PMANM',         # cpsG — mannose precursor
    'RHCCE',         # luxS — quorum sensing / AI-2
    'AI2K',
]

def run_eps_analysis(base_model, medium, biomass_fraction=0.9):
    m = base_model.copy()
    m.medium = medium

    # Stage 1: maximize growth
    m.objective = BIOMASS_RXN
    sol_growth = m.optimize()
    mu_max = sol_growth.objective_value

    # Stage 2: constrain growth, maximize colanic acid secretion
    biomass_rxn = m.reactions.get_by_id(BIOMASS_RXN)
    biomass_rxn.lower_bound = biomass_fraction * mu_max
    m.objective = EPS_RXN
    sol_eps = m.optimize()

    result = {
        'mu_max':         mu_max,
        'biomass_at_eps': sol_eps.fluxes[BIOMASS_RXN],
    }
    for rxn_id in EPS_RXNS:
        result[rxn_id] = sol_eps.fluxes[rxn_id]

    return result

## Run Both Conditions

In [28]:
m9_results = run_eps_analysis(model, m9_medium)
lb_results = run_eps_analysis(model, lb_medium)

print('M9 results:', m9_results)
print('LB results:', lb_results)

M9 results: {'mu_max': 0.8217947382569741, 'biomass_at_eps': np.float64(0.7396152644312768), 'UDPGPT': np.float64(0.2627443914851906), 'COLASYNTH': np.float64(0.26274439148519063), 'EX_colacid_e': np.float64(0.2627443914851906), 'CELSYNTH': np.float64(0.0), 'EX_cellulose_e': np.float64(0.0), 'EX_puacgam_e': np.float64(0.0), 'PMANM': np.float64(-0.5254887829703812), 'RHCCE': np.float64(0.00033134763846521197), 'AI2K': np.float64(0.00033134763846521197)}
LB results: {'mu_max': 1.9527099327190323, 'biomass_at_eps': np.float64(1.757438939447129), 'UDPGPT': np.float64(0.4758855127317061), 'COLASYNTH': np.float64(0.47588551273170615), 'EX_colacid_e': np.float64(0.47588551273170604), 'CELSYNTH': np.float64(0.0), 'EX_cellulose_e': np.float64(0.0), 'EX_puacgam_e': np.float64(0.0), 'PMANM': np.float64(-0.951771025463412), 'RHCCE': np.float64(0.0007838177669831932), 'AI2K': np.float64(0.0007838177669831923)}


## Comparison Table

In [29]:
comparison = pd.DataFrame({
    'M9 Glucose': m9_results,
    'LB Medium':  lb_results
})

comparison.index.name = 'metric'
comparison['fold_change (LB/M9)'] = (comparison['LB Medium'] / comparison['M9 Glucose']).round(2)

print(comparison.to_string())

                M9 Glucose  LB Medium  fold_change (LB/M9)
metric                                                    
mu_max            0.821795   1.952710                 2.38
biomass_at_eps    0.739615   1.757439                 2.38
UDPGPT            0.262744   0.475886                 1.81
COLASYNTH         0.262744   0.475886                 1.81
EX_colacid_e      0.262744   0.475886                 1.81
CELSYNTH          0.000000   0.000000                  NaN
EX_cellulose_e    0.000000   0.000000                  NaN
EX_puacgam_e      0.000000   0.000000                  NaN
PMANM            -0.525489  -0.951771                 1.81
RHCCE             0.000331   0.000784                 2.37
AI2K              0.000331   0.000784                 2.37


## Notes on EPS Components

### Why CELSYNTH = 0
Cellulose synthesis requires c-di-GMP as a cofactor. FBA does not produce c-di-GMP during normal growth because it is not required for biomass. Cellulose is more relevant to mature biofilm architecture, not initial attachment, and is naturally low in E. coli K-12 strains. To activate cellulose flux, a biofilm-inducing state must be simulated by knocking out the c-di-GMP degradation reactions (CDGUNPD, LDGUNPD) to allow c-di-GMP to accumulate.

### Do we need PNAG?
Depends on the experimental assay:
- **Bulk polysaccharide assay** (e.g. phenol-sulfuric acid, crystal violet): PNAG should be included as it contributes to total EPS.
- **Colanic acid-specific assay** (e.g. fucose-based colorimetric): colanic acid alone is sufficient.

In E. coli K-12 (iML1515), colanic acid is the dominant secreted EPS. PNAG is partially retained at the cell surface as an adhesin rather than fully secreted, so it may not appear in supernatant-based EPS measurements. `EX_colacid_e` is the primary EPS proxy in this model. PNAG (`EX_puacgam_e`) is present and can be activated by changing the optimization objective if needed.